In [4]:
from langgraph.graph import StateGraph,START , END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [22]:
class BlogState(TypedDict):

  title:str
  outline:str
  content:str
  evaluation:str
  marks:int

In [23]:
def create_outline(state: BlogState) -> BlogState:

  # fetch title
  title = state['title']

  # call llm gen outline
  prompt = f"Generate an detailed outline for a blog on the topic - {title}"
  outline = model.invoke(prompt).content

  # update state
  state['outline'] = outline

  return state

In [24]:
def create_blog(state:BlogState)-> BlogState:
  title = state['title']
  outline = state['outline']

  prompt = f"Write a detailed blog on the title - {title} using the following outline \n {outline}"

  content = model.invoke(prompt).content

  state['content'] = content

  return state

In [25]:
def evaluate_blog(state:BlogState)->BlogState:
  title = state['title']
  content = state['content']

  prompt = f"""
    Evaluate the following blog.

    Blog title: {title}

    Blog content:
    {content}

    Evaluate using these criteria:
    1. Relevance to the title: 2 marks
    2. Structure and organization: 2 marks
    3. Depth and completeness: 2 marks
    4. Readability and grammar: 2 marks
    5. Factual accuracy: 2 marks                                     
    Give a total score out of 10.

    Return your response in exactly this format:

    Marks: <integer out of 10>
    Feedback: <brief explanation of strengths and weaknesses>
    """

  response = model.invoke(prompt).content

  # Extract the marks from the evaluation
  import re

  match = re.search(r"Marks:\s*(\d+)", response)

  marks = int(match.group(1)) if match else 0

  # Keep marks within the valid range
  marks = max(0, min(10,marks))

  state['evaluation'] = response
  state['marks'] = marks

  return state

In [26]:
graph = StateGraph(BlogState)

#nodes 
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog',evaluate_blog)

# edges 
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [27]:
initial_state = {'title':'Rise of AI in India'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': '**Blog Title:** *The Rise of AI in India: Opportunities, Challenges, and the Road Ahead*\n\n---\n\n## 1. Introduction  \n- **Hook:** A striking statistic or anecdote (e.g., “India’s AI market is projected to hit $17\u202fbillion by 2027, outpacing many developed economies.”)  \n- **Why It Matters:** Briefly explain AI’s global significance and why India’s trajectory is a game‑changer for the world.  \n- **Thesis Statement:** Outline the blog’s purpose – to map the evolution of AI in India, highlight key drivers, showcase success stories, discuss challenges, and envision the future.\n\n---\n\n## 2. Historical Context: From Early Adoption to Mainstream Momentum  \n### 2.1. The Foundations (1990‑2010)  \n- Early research hubs (IITs, ISI, IISc) and government-funded labs.  \n- First AI patents and publications from Indian institutions.  \n\n### 2.2. The Turning Point (2010‑2015)  \n- Launch of **Digital India** and **National Knowledge Network**

In [31]:
print(final_state['outline'])

**Blog Title:** *The Rise of AI in India: Opportunities, Challenges, and the Road Ahead*

---

## 1. Introduction  
- **Hook:** A striking statistic or anecdote (e.g., “India’s AI market is projected to hit $17 billion by 2027, outpacing many developed economies.”)  
- **Why It Matters:** Briefly explain AI’s global significance and why India’s trajectory is a game‑changer for the world.  
- **Thesis Statement:** Outline the blog’s purpose – to map the evolution of AI in India, highlight key drivers, showcase success stories, discuss challenges, and envision the future.

---

## 2. Historical Context: From Early Adoption to Mainstream Momentum  
### 2.1. The Foundations (1990‑2010)  
- Early research hubs (IITs, ISI, IISc) and government-funded labs.  
- First AI patents and publications from Indian institutions.  

### 2.2. The Turning Point (2010‑2015)  
- Launch of **Digital India** and **National Knowledge Network**.  
- Arrival of global tech giants (Google, Microsoft, IBM) establ

In [32]:
print(final_state['content'])

# The Rise of AI in India: Opportunities, Challenges, and the Road Ahead  

*By [Your Name] – [Date]*  

---  

## 1. Introduction  

> **“India’s artificial‑intelligence market is projected to hit **$17 billion** by 2027, outpacing many developed economies.”** – NITI Aayog, 2023  

Artificial Intelligence (AI) is no longer a futuristic buzzword; it is reshaping economies, societies, and daily life across the globe. For a country of 1.4 billion people, a burgeoning tech talent pool, and a rapidly digitising public sector, the AI story is especially compelling.  

**Why it matters:**  

* **Economic multiplier:** AI could add up to $1 trillion to India’s GDP by 2035, according to a McKinsey estimate.  
* **Social impact:** From early‑disease detection to climate‑smart farming, AI has the power to address India’s most pressing development challenges.  
* **Geopolitical relevance:** As the world races to set AI standards, India’s policy choices will influence the global governance landsca

In [30]:
print(final_state['evaluation'])

Marks: 9  
Feedback: The blog aligns tightly with its title, offering a clear, well‑organized progression from history to drivers, sector impacts, success stories, and challenges. Its depth is strong, covering multiple industries with concrete examples and data. The writing is generally clear and grammatically sound, making it easy to read. Most facts are accurate and sourced, though a few statistics (e.g., exact GDP contribution or AI‑related patent counts) could benefit from explicit citations, which prevents a perfect score on factual precision. Overall, it is a comprehensive and engaging piece on AI’s rise in India.
